In [20]:
import rmllm
import psychscanner as psy
import json
import copy


rm2_sc_st = psy.get_task_template()
rm2_2op = rm2_sc_st.copy()

In [21]:
rm2_2op["taskname"] = "rm_2op_convo_nofb"
rm2_2op["contexts"] = ["perceived","imagined","test:perceived","test:imagined"]
rm2_2op["context_present"] = False
rm2_2op["chain_type"] = "task"
rm2_2op["tasktype"] = "episodic_system"
rm2_2op["contexts_id"] = ["perceived","imagined","test:perceived","test:imagined"]
rm2_2op["parser"] = "TwoResponses"
rm2_2op["instructions"] = {"definition":""}

In [22]:
task_prep_dir = rmllm.config.RAW_DATA_DIR/"prepare_task"
with open(task_prep_dir/"task_rm_v1.json","r") as f:
    taskbase_2op = json.load(f)

with open(task_prep_dir/"rm_task_2op_inst.json") as f:
    instr_2op = json.load(f)

In [23]:
part_1_instructions = {
    "task_definition": [
        'You are a helpful participant performing a task with two different components for successful response.',
        'Each component refers to a unique problem about the task as described in the instructions below.',
        "Two components of the task are: ['WORD_PAIR_TASK', 'RELATEDNESS_RATING_BETWEEN_WORD_1_AND_WORD_2']",
        'Follow all the instructions related to different components of the task to give accurate response.',
        '**COMMITMENT** You have made the commitment to make sure to follow all the the instructions for each of the components and formatting your task response.'
    ],
    "INSTRUCTIONS":{
        'WORD_PAIR_TASK': {
            '**Identify the Word Pair**': "Look for the word pair, referred to as 'word_1' and 'word_2'.",
            "**Check if 'word_2' is Provided**": [
                "If 'word_2' is a complete and valid English word, report that word as it is.",
                "If 'word_2' contains a blank value ('_______'), proceed to the next step."],
                  "**If 'word_2' is a blank in the word-pair then imagine**": ["**Replace blank value of 'word_2' by using your an imagination to create **an** english word to complete the word pair.**"],
                  'You are **prohibted** to imagine': ["Use of given 'word_1' in any form.",
                                             'Compound words made of smaller word,',
                                             'Symbols and numbers like: _/^(?=.*?[1-9])[0-9()-]+$,',
                                             'anything outside of the words would be not in a english dictionary.',
                                             'Mathmetical or algebraic equations'],
                   'You are **allowed** to imagine': 'singular english words that are **not** prohibited as described above.'
        },
        'RELATEDNESS_RATING_BETWEEN_WORD_1_AND_WORD_2': {
        'definition': ["Numeric value in between 0 (not realated at all) and 100 (very highly related) for Relatedness between two words similarity between the values of 'word_1' and 'word_2'.",
        'It can be based on mutiple factors,for example:',
        'similar sounding (phonetics),',
        'meaning (semantics),',
        'or categorical (whether the two words can related through common categories).',
        ''],
        'relatedness_rating_scale': ['Use any value in the range of 0 to 100 in relatedness rating scale.',
                                     '**0** signifies the words are **not at all related**.',
                                     '**100** signifies the words **very highly related**.',
                                     'Intermediate values between 0 and 100 denote intermediate values.','Relateness value is **stricity defined in the **range of 0: not related at all, TO 100: very highly related**.',
                                     '**ALWAYS FOLLOW THESE INSTRUCTION**'],
        'type': 'number'
        },
    }
}


In [24]:
part_2_instructions_order1 = {
    "task_definition": [
        "You are a helpful participant performing a task based on **previously completed word-pair task** with two different components for successful response.",
        "Here, you will be asked to make judgements about the word you reported as **word_2** in the previous task based on the **Test_Word**",
        "Two components of the current task are: ['JUDGE_GENERATION_TYPE_OF_WORD_2', 'CONFIDENCE_ON_JUDGEMENT_OF_GENERATION_TYPE']",
        "Follow all the instructions related to different components of the task to give accurate response.",
        "**COMMITMENT** You have made the commitment to make sure to follow all the the instructions for each of the components and formatting your task response."
    ],
    "INSTRUCTIONS": {
        "JUDGE_GENERATION_TYPE_OF_WORD_2": {
            "definition": [
                "During the word-pair task, **Test_Word** would have appeared as **Word_1** in one of the pair for which you reported the same word as **word_2** or imagined it.",
                "Judge the nature of generation of the value of 'word_2' on following options:"
            ],
            "options": {
                "enum": [
                    "external",
                    "internal"
                ]
            },
            "option_descriptions": {
                "external": "**Only If** the 'word_2' was **provided in the task as an english word.**.",
                "internal": "**Only If** the 'word_2' was **imagined by you to complete the word-pair and replace the blank.**"
            }
        },
        "CONFIDENCE_ON_JUDGEMENT_OF_GENERATION_TYPE": {
            "definition": [
                "Report the **confidence level** about the **selected option** for the judgment of generation type of the 'word_2'.",
                "**Confidence level** indexes your ability **to know** the **level of certainty** that your reported **generation judgment is correct**.",
                "In the context of this task, it is your ability to report **level of certainty** in your judgments about the the generation type for the value of 'word_2'.",
                "Use the confidence rating scale faithfully to report the **level of certainty** as your confidence rating in the ordinal confidence scale described below.",
                "The ordinal values of the confidence scale from 1 to 6.",
                "Higher the numberic value of confidence, higher is your confidence level in the judgment about the generation type being correct.",
                "**Follow the instructions and confidence scale be able to able to be able to faithfully report the confidence level in your selected generation type for the value of 'word_2'."
            ],
            "enum": [
                1,
                2,
                3,
                4,
                5,
                6
            ],
            "confidence_scale": {
                "**1**": "**Not at all confident.**",
                "**2**": "**Slightly confident.**",
                "**3**": "**Moderately confident.**",
                "**4**": "**Fairly confident.**",
                "**5**": "**Very confident.**",
                "**6**": "**Highly confident.**"
            },
            "type": "number"
        }
    }
}




part_2_instructions_order2 = {
    "task_definition": [
        "You are a helpful participant performing a task based on **previously completed word-pair task** with two different components for successful response.",
        "Here, you will be asked to make judgements about the word you reported as **word_2** in the previous task based on the **Test_Word**",
        "Two components of the current task are: ['JUDGE_GENERATION_TYPE_OF_WORD_2', 'CONFIDENCE_ON_JUDGEMENT_OF_GENERATION_TYPE']",
        "Follow all the instructions related to different components of the task to give accurate response.",
        "**COMMITMENT** You have made the commitment to make sure to follow all the the instructions for each of the components and formatting your task response."
    ],
    "INSTRUCTIONS": {
        "JUDGE_GENERATION_TYPE_OF_WORD_2": {
            "definition": [
                "During the word-pair task, **Test_Word** would have appeared as **Word_1** in one of the pair for which you reported the same word as **word_2** or imagined it.",
                "Judge the nature of generation of the value of 'word_2' on following options:"
            ],
            "options": {
                "enum": [
                    "internal",
                    "external"
                ]
            },
            "option_descriptions": {
                "internal": "**Only If** the 'word_2' was **imagined by you to complete the word-pair and replace the blank.**",
                "external": "**Only If** the 'word_2' was **provided in the task as an english word.**."
            }
        },
        "CONFIDENCE_ON_JUDGEMENT_OF_GENERATION_TYPE": {
            "definition": [
                "Report the **confidence level** about the **selected option** for the judgment of generation type of the 'word_2'.",
                "**Confidence level** indexes your ability **to know** the **level of certainty** that your reported **generation judgment is correct**.",
                "In the context of this task, it is your ability to report **level of certainty** in your judgments about the the generation type for the value of 'word_2'.",
                "Use the confidence rating scale faithfully to report the **level of certainty** as your confidence rating in the ordinal confidence scale described below.",
                "The ordinal values of the confidence scale from 1 to 6.",
                "Higher the numberic value of confidence, higher is your confidence level in the judgment about the generation type being correct.",
                "**Follow the instructions and confidence scale be able to able to be able to faithfully report the confidence level in your selected generation type for the value of 'word_2'."
            ],
            "enum": [
                1,
                2,
                3,
                4,
                5,
                6
            ],
            "confidence_scale": {
                "**1**": "**Not at all confident.**",
                "**2**": "**Slightly confident.**",
                "**3**": "**Moderately confident.**",
                "**4**": "**Fairly confident.**",
                "**5**": "**Very confident.**",
                "**6**": "**Highly confident.**"
            },
            "type": "number"
        }
    }
}

In [25]:
items_part_1 = {}
for i, j in enumerate(taskbase_2op["task1"]["trials"]):
    ttype = j["source"].split(" ")[-1].lower()
    tr_code = f"{ttype}_{i+1}"
    items_part_1[tr_code] = []
    items_part_1[tr_code].append(
        {
            "stimulus": {
                "Word_Pair": {'word_1': j['stim']['word 1'],
                'word_2': j['stim']['word 2']},  },
                "trcode": tr_code
            }
        )
source_corrans_map = {"imagined":"internal","perceived":"external"}

def sample_stims(items_set, n=10):
    itemssample = {}
    img_count = 0
    per_count = 0
    for i in items_set:
        if "perceived" in i:
            per_count += 1
            if per_count <= n:
                itemssample[i] = items_set[i]
        if "imagined" in i:
            img_count += 1
            if img_count <= n:
                itemssample[i] = items_set[i]  

    return itemssample  

def create_part2_on_part1(items_set):
    items_part_2 = {}
    for i,j in items_set.items():
        items_part_2["test:"+i] = [{'stimulus': {'Test_Word':j[0]['stimulus']['Word_Pair']['word_1']},
                            'trcode': "test:"+j[0]['trcode'], 'corrAns': source_corrans_map[i.split("_")[0]],
                            'og_source':j[0]['stimulus']}]
        
    return items_part_2    


In [26]:
def add_system_msg_to_trials( trials, system_msg):
    for t,x in trials.items():
        x[0]['system_message'] = system_msg
    return trials


In [27]:
items10_part_1 = sample_stims(items_part_1, n=10)
items10_part_2 = create_part2_on_part1(items10_part_1)
items10_part_1_with_inst = add_system_msg_to_trials(items10_part_1,part_1_instructions)
items10_part_2_with_inst1 = add_system_msg_to_trials(items10_part_2,part_2_instructions_order1)
items10_part_2_with_inst1_r = add_system_msg_to_trials(items10_part_2,part_2_instructions_order2)

items5_part_1 = sample_stims(items_part_1, n=5)
items5_part_2 = create_part2_on_part1(items5_part_1)
items5_part_1_with_inst = add_system_msg_to_trials(items5_part_1,part_1_instructions)
items5_part_2_with_inst1 = add_system_msg_to_trials(items5_part_2,part_2_instructions_order1)
items5_part_2_with_inst1_r = add_system_msg_to_trials(items5_part_2,part_2_instructions_order2)


In [28]:
all_items10 = {**items10_part_1_with_inst, **items10_part_2_with_inst1}
all_items10_r = {**items10_part_1_with_inst, **items10_part_2_with_inst1_r}

all_items5 = {**items5_part_1_with_inst, **items5_part_2_with_inst1}
all_items5_r = {**items5_part_1_with_inst, **items5_part_2_with_inst1_r}

In [29]:
# save files
def savejson(dir,name,data):
    with open(dir/name,"w") as f:
        json.dump(data,f,indent=4)
OUT_DIR = rmllm.config.RAW_DATA_DIR/"runtasks2"
if not OUT_DIR.is_dir():
    OUT_DIR.mkdir()

In [30]:
OUT_DIR = rmllm.config.RAW_DATA_DIR/"runtasks2"
rm2_2op["items"] = all_items10
rm2_2op["taskname"] = "rm_2op_convo"
savejson(OUT_DIR,"rm_2op_convo.json",rm2_2op)

rm2_2op["items"] = all_items10_r
rm2_2op["taskname"] = "rm_2op_convo_r"
savejson(OUT_DIR,"rm_2op_convo_r.json",rm2_2op)

rm2_2op["items"] = all_items5
rm2_2op["taskname"] = "rm_2op_5t_convo"
savejson(OUT_DIR,"rm_2op_5t_convo.json",rm2_2op)

rm2_2op["items"] = all_items5_r
rm2_2op["taskname"] = "rm_2op_5t_convo_r"
savejson(OUT_DIR,"rm_2op_5t_convo_r.json",rm2_2op)